# CoralGuard-X — FINAL Hybrid Coral AI Pipeline

```text
Underwater Image / Video
        ↓
YOLO26m High-Recall Candidate Detection
        ↓
CoralSCOP Box-Prompted Coral Mask Refinement
        ↓
Mask-IoU Duplicate / Overlap Resolver
        ↓
Dual-View EfficientNetV2-S
  ├─ masked coral foreground
  └─ +10% context crop
        ↓
6 known species + NON_CORAL
        ↓
Embedding / confidence OOD gate
  ├─ known species
  ├─ Other / Unknown Coral
  └─ Non-Coral
        ↓
SAM 2.1 Video Memory
        ↓
Stable masks + IDs + species EMA
        ↓
HQ final video
```

The notebook automatically prefers the strongest checkpoints already present in:

```text
/content/drive/MyDrive/inferance/
```

Detector priority:
`best_yolo26m_plus150.pt` → `best_yolo26m_extended.pt` → `best.pt`

Classifier priority:
`best_efficientnetv2s_plus150.pth` → `best_efficientnetv2s_macro_f1.pth`

**Important:** the current dataset does not explicitly label `Dead Coral` or `Algae`, so this notebook does not invent those labels. Those cases are routed to `Non-Coral` or `Other/Unknown Coral` unless you later add explicit labels.

The held-out detector test split is not used for training or threshold tuning.


In [ ]:
# CELL 1 — INSTALL + DRIVE + GPU

!pip -q install ultralytics==8.4.117 timm==1.0.22 scikit-learn scipy pandas pycocotools huggingface_hub
!rm -rf /content/CoralSCOP
!git clone -q https://github.com/zhengziqiang/CoralSCOP /content/CoralSCOP
!pip -q install git+https://github.com/facebookresearch/sam2.git

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from collections import defaultdict
import os, sys, gc, csv, json, math, random, shutil, time, zipfile, subprocess, requests

import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import InterpolationMode

import timm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from scipy.optimize import linear_sum_assignment
from ultralytics import YOLO
import ultralytics

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
if not torch.cuda.is_available(): raise RuntimeError("Enable a GPU runtime.")

DEVICE=torch.device("cuda")
GPU_NAME=torch.cuda.get_device_name(0)
VRAM_GB=torch.cuda.get_device_properties(0).total_memory/(1024**3)
CLS_BATCH=96 if VRAM_GB>=70 else 64 if VRAM_GB>=38 else 32 if VRAM_GB>=22 else 16 if VRAM_GB>=14 else 8

print("GPU:",GPU_NAME)
print("VRAM GB:",round(VRAM_GB,2))
print("Torch:",torch.__version__)
print("Ultralytics:",ultralytics.__version__)
print("timm:",timm.__version__)
print("CLS_BATCH:",CLS_BATCH)
print("STATUS: ENVIRONMENT_READY")


In [ ]:
# CELL 2 — PATHS + AUTOMATIC CHECKPOINT SELECTION

MYDRIVE=Path('/content/drive/MyDrive')
PROJECT_ROOT=MYDRIVE/'Coral_Reef_AI_Project'
DET_ROOT=PROJECT_ROOT/'01_Data'/'Final'/'Detection_v1'

TRAIN_IMG=DET_ROOT/'images'/'train'
VAL_IMG=DET_ROOT/'images'/'validation'
TEST_IMG=DET_ROOT/'images'/'test'
TRAIN_COCO=DET_ROOT/'annotations'/'instances_train.json'
VAL_COCO=DET_ROOT/'annotations'/'instances_validation.json'
MANIFEST=PROJECT_ROOT/'04_Dataset_Splits'/'classification_roi_manifest_split.csv'

INFER_ROOT=MYDRIVE/'inferance'
FINAL_ROOT=INFER_ROOT/'CoralGuard_X_FINAL'
MASK_ROOT=FINAL_ROOT/'01_CoralSCOP_Masks'
DUAL_ROOT=FINAL_ROOT/'02_DualView_Data'
CLS_ROOT=FINAL_ROOT/'03_DualView_Classifier'
CAL_ROOT=FINAL_ROOT/'04_OOD_Calibration'
VIDEO_ROOT=FINAL_ROOT/'05_Final_Videos'
for d in [FINAL_ROOT,MASK_ROOT,DUAL_ROOT,CLS_ROOT,CAL_ROOT,VIDEO_ROOT]:
    d.mkdir(parents=True,exist_ok=True)

def first_existing(paths):
    for p in paths:
        if p.exists(): return p
    return None

DETECTOR_CKPT=first_existing([
    INFER_ROOT/'best_yolo26m_plus150.pt',
    INFER_ROOT/'best_yolo26m_extended.pt',
    INFER_ROOT/'best.pt',
])
OLD_CLS_CKPT=first_existing([
    INFER_ROOT/'best_efficientnetv2s_plus150.pth',
    INFER_ROOT/'best_efficientnetv2s_macro_f1.pth',
])

if DETECTOR_CKPT is None: raise FileNotFoundError("No YOLO checkpoint found in MyDrive/inferance")
if OLD_CLS_CKPT is None: raise FileNotFoundError("No EfficientNetV2-S checkpoint found in MyDrive/inferance")

for p in [TRAIN_IMG,VAL_IMG,TRAIN_COCO,VAL_COCO,MANIFEST,DETECTOR_CKPT,OLD_CLS_CKPT]:
    if not p.exists(): raise FileNotFoundError(p)

print("Detector:",DETECTOR_CKPT)
print("Classifier init:",OLD_CLS_CKPT)
print("Output:",FINAL_ROOT)
print("Held-out TEST: untouched")
print("STATUS: PATHS_READY")


In [ ]:
# CELL 3 — OFFICIAL CORALSCOP CHECKPOINT + BOX-PROMPT REFINER

CORALSCOP_CKPT=INFER_ROOT/'vit_b_coralscop.pth'
CORALSCOP_URL=(
    'https://www.dropbox.com/scl/fi/pw5jiq9oc8e8kvkx1fdk0/vit_b_coralscop.pth'
    '?rlkey=qczdohnzxwgwoadpzeht0lim2&dl=1'
)

if not CORALSCOP_CKPT.exists():
    print("Downloading official CoralSCOP ViT-B checkpoint...")
    with requests.get(CORALSCOP_URL,stream=True,timeout=90) as r:
        r.raise_for_status()
        with open(CORALSCOP_CKPT,'wb') as f:
            for chunk in r.iter_content(1024*1024):
                if chunk: f.write(chunk)

if CORALSCOP_CKPT.stat().st_size < 50*1024*1024:
    raise RuntimeError("CoralSCOP checkpoint looks incomplete.")

sys.path.insert(0,'/content/CoralSCOP')
from segment_anything import sam_model_registry, SamPredictor

coralscop_model=sam_model_registry['vit_b'](checkpoint=str(CORALSCOP_CKPT))
coralscop_model.to(DEVICE).eval()
coralscop_predictor=SamPredictor(coralscop_model)

MASK_MIN_SCORE=0.35
MASK_MIN_PIXELS=40
MASK_DUP_IOU=0.65

def box_area(b):
    x1,y1,x2,y2=map(float,b)
    return max(0.,x2-x1)*max(0.,y2-y1)

def box_iou(a,b):
    a=np.asarray(a,np.float32); b=np.asarray(b,np.float32)
    xx1=max(a[0],b[0]); yy1=max(a[1],b[1]); xx2=min(a[2],b[2]); yy2=min(a[3],b[3])
    inter=max(0.,xx2-xx1)*max(0.,yy2-yy1)
    return inter/max(box_area(a)+box_area(b)-inter,1e-9)

def mask_iou(a,b):
    inter=np.logical_and(a,b).sum()
    union=np.logical_or(a,b).sum()
    return float(inter/max(union,1))

def mask_bbox(mask):
    ys,xs=np.where(mask)
    if len(xs)==0: return None
    return np.array([xs.min(),ys.min(),xs.max()+1,ys.max()+1],dtype=np.float32)

def coralscop_prompt(rgb,box):
    coralscop_predictor.set_image(rgb)
    with torch.inference_mode():
        masks,scores,_=coralscop_predictor.predict(
            box=np.asarray(box,dtype=np.float32),
            multimask_output=True
        )
    order=np.argsort(scores)[::-1]
    for idx in order:
        m=masks[idx].astype(bool)
        if m.sum()>=MASK_MIN_PIXELS and float(scores[idx])>=MASK_MIN_SCORE:
            return m,float(scores[idx])
    idx=int(order[0])
    return masks[idx].astype(bool),float(scores[idx])

print("CoralSCOP:",CORALSCOP_CKPT)
print("STATUS: CORALSCOP_READY")


In [ ]:
# CELL 4 — GENERATE / RESUME CORALSCOP MASKS FROM GT BOXES

MASK_MANIFEST=MASK_ROOT/'coralscop_mask_manifest.csv'

def resolve_image(folder,file_name):
    p=folder/Path(file_name).name
    if p.exists(): return p
    matches=list(folder.glob(Path(file_name).stem+'.*'))
    if not matches: raise FileNotFoundError(file_name)
    return matches[0]

existing={}
if MASK_MANIFEST.exists():
    old=pd.read_csv(MASK_MANIFEST)
    for _,r in old.iterrows():
        existing[(str(r['split']),int(r['ann_id']))]=dict(r)

rows=[]

for split,coco_path,img_folder in [
    ('train',TRAIN_COCO,TRAIN_IMG),
    ('validation',VAL_COCO,VAL_IMG),
]:
    data=json.loads(coco_path.read_text(encoding='utf-8'))
    by_img=defaultdict(list)
    for a in data['annotations']: by_img[int(a['image_id'])].append(a)
    info={int(im['id']):im for im in data['images']}
    out_dir=MASK_ROOT/split
    out_dir.mkdir(parents=True,exist_ok=True)

    for n,(image_id,im) in enumerate(info.items(),1):
        anns=by_img[image_id]
        if all((split,int(a['id'])) in existing for a in anns):
            rows.extend(existing[(split,int(a['id']))] for a in anns)
            continue

        path=resolve_image(img_folder,im['file_name'])
        rgb=np.asarray(Image.open(path).convert('RGB'))
        H,W=rgb.shape[:2]
        coralscop_predictor.set_image(rgb)

        for a in anns:
            key=(split,int(a['id']))
            if key in existing:
                rows.append(existing[key]); continue

            x,y,w,h=map(float,a['bbox'])
            box=np.array([x,y,x+w,y+h],dtype=np.float32)
            box[0::2]=np.clip(box[0::2],0,W-1)
            box[1::2]=np.clip(box[1::2],0,H-1)

            with torch.inference_mode():
                masks,scores,_=coralscop_predictor.predict(box=box,multimask_output=True)

            order=np.argsort(scores)[::-1]
            chosen=None; chosen_score=0.0
            for idx in order:
                m=masks[idx].astype(bool)
                if m.sum()>=MASK_MIN_PIXELS and float(scores[idx])>=MASK_MIN_SCORE:
                    chosen=m; chosen_score=float(scores[idx]); break

            if chosen is None:
                idx=int(order[0])
                chosen=masks[idx].astype(bool)
                chosen_score=float(scores[idx])

            mask_path=out_dir/f"ann_{int(a['id']):06d}.png"
            cv2.imwrite(str(mask_path),chosen.astype(np.uint8)*255)

            row={
                'split':split,
                'ann_id':int(a['id']),
                'image_id':image_id,
                'image_path':str(path),
                'image_stem':path.stem,
                'mask_path':str(mask_path),
                'x1':float(box[0]),'y1':float(box[1]),
                'x2':float(box[2]),'y2':float(box[3]),
                'mask_score':chosen_score,
            }
            rows.append(row); existing[key]=row

        pd.DataFrame(rows).drop_duplicates(['split','ann_id']).to_csv(MASK_MANIFEST,index=False)
        if n%25==0: print(f"{split}: {n}/{len(info)}")

mask_df=pd.read_csv(MASK_MANIFEST)
print(mask_df.groupby('split').size())
print("Mean CoralSCOP score:",float(mask_df['mask_score'].mean()))
print("STATUS: MASKS_READY")


In [ ]:
# CELL 5 — MAP SPECIES + BUILD MASKED/CONTEXT POSITIVES AND REAL BACKGROUND NEGATIVES

roi=pd.read_csv(MANIFEST)
needed={'roi_id','image_stem','raw_detection_label','x1_px','y1_px','x2_px','y2_px','split'}
if needed-set(roi.columns):
    raise RuntimeError("Classification manifest schema mismatch.")

mask_index=defaultdict(list)
for _,r in mask_df.iterrows():
    mask_index[(str(r['split']),str(r['image_stem']))].append(r)

positive_rows=[]
for _,r in roi[roi['split'].astype(str).isin(['train','validation'])].iterrows():
    key=(str(r['split']),str(r['image_stem']))
    candidates=mask_index[key]
    q=[float(r.x1_px),float(r.y1_px),float(r.x2_px),float(r.y2_px)]
    best=max(candidates,key=lambda z:box_iou(q,[z.x1,z.y1,z.x2,z.y2]))
    if box_iou(q,[best.x1,best.y1,best.x2,best.y2])<0.90:
        raise RuntimeError(f"Mask mapping failed for {r.roi_id}")
    positive_rows.append({
        'roi_id':str(r.roi_id),
        'split':str(r['split']),
        'label':str(r.raw_detection_label),
        'image_path':str(best.image_path),
        'mask_path':str(best.mask_path),
        'x1':q[0],'y1':q[1],'x2':q[2],'y2':q[3],
    })
pos=pd.DataFrame(positive_rows)
assert (pos['split']=='train').sum()==1540
assert (pos['split']=='validation').sum()==354

CONTEXT_PAD=0.10
PAIR_MANIFEST=DUAL_ROOT/'dual_view_manifest.csv'

def padded_box(box,W,H,pad=0.10):
    x1,y1,x2,y2=map(float,box)
    bw=max(1.,x2-x1); bh=max(1.,y2-y1)
    px=bw*pad; py=bh*pad
    return (
        max(0,int(math.floor(x1-px))),max(0,int(math.floor(y1-py))),
        min(W,int(math.ceil(x2+px))),min(H,int(math.ceil(y2+py)))
    )

pair_rows=[]
for i,r in pos.iterrows():
    out_dir=DUAL_ROOT/r['split']/r['label']; out_dir.mkdir(parents=True,exist_ok=True)
    cpath=out_dir/f"{r.roi_id}__context.jpg"
    mpath=out_dir/f"{r.roi_id}__masked.jpg"

    if not (cpath.exists() and mpath.exists()):
        rgb=np.asarray(Image.open(r.image_path).convert('RGB'))
        mask=cv2.imread(r.mask_path,cv2.IMREAD_GRAYSCALE)>127
        H,W=rgb.shape[:2]
        x1,y1,x2,y2=padded_box([r.x1,r.y1,r.x2,r.y2],W,H,CONTEXT_PAD)
        context=rgb[y1:y2,x1:x2].copy()
        local_mask=mask[y1:y2,x1:x2]
        masked=context.copy(); masked[~local_mask]=0
        Image.fromarray(context).save(cpath,'JPEG',quality=95,subsampling=0)
        Image.fromarray(masked).save(mpath,'JPEG',quality=95,subsampling=0)

    pair_rows.append({'split':r['split'],'label':r['label'],'context_path':str(cpath),'masked_path':str(mpath),'source':'CORAL'})

# Sample one realistic NON_CORAL crop/image outside GT boxes.
def coco_boxes(coco_path):
    data=json.loads(coco_path.read_text(encoding='utf-8'))
    info={int(im['id']):im for im in data['images']}
    boxes=defaultdict(list)
    for a in data['annotations']:
        x,y,w,h=map(float,a['bbox'])
        boxes[int(a['image_id'])].append([x,y,x+w,y+h])
    return info,boxes

rng=np.random.default_rng(SEED)
for split,coco_path,img_folder in [('train',TRAIN_COCO,TRAIN_IMG),('validation',VAL_COCO,VAL_IMG)]:
    info,boxes_by_img=coco_boxes(coco_path)
    out_dir=DUAL_ROOT/split/'NON_CORAL'; out_dir.mkdir(parents=True,exist_ok=True)
    for image_id,im in info.items():
        path=resolve_image(img_folder,im['file_name'])
        rgb=np.asarray(Image.open(path).convert('RGB')); H,W=rgb.shape[:2]
        q=None
        for _ in range(120):
            side=int(max(48,min(W,H)*float(rng.uniform(.10,.32))))
            if W<=side or H<=side: continue
            x1=int(rng.integers(0,W-side)); y1=int(rng.integers(0,H-side))
            cand=[x1,y1,x1+side,y1+side]
            mx=max([box_iou(cand,b) for b in boxes_by_img[image_id]]+[0.])
            if mx<0.02: q=cand; break
        if q is None: continue
        x1,y1,x2,y2=q
        context=rgb[y1:y2,x1:x2].copy()
        masked=np.zeros_like(context)
        stem=Path(im['file_name']).stem
        cpath=out_dir/f"{stem}__neg__context.jpg"
        mpath=out_dir/f"{stem}__neg__masked.jpg"
        Image.fromarray(context).save(cpath,'JPEG',quality=95,subsampling=0)
        Image.fromarray(masked).save(mpath,'JPEG',quality=95,subsampling=0)
        pair_rows.append({'split':split,'label':'NON_CORAL','context_path':str(cpath),'masked_path':str(mpath),'source':'BACKGROUND'})

pair_df=pd.DataFrame(pair_rows)
pair_df.to_csv(PAIR_MANIFEST,index=False)
print(pair_df.groupby(['split','label']).size())
print("STATUS: DUAL_VIEW_DATA_READY")


In [ ]:
# CELL 6 — FINAL DUAL-VIEW EFFICIENTNETV2-S + TRANSFER LEARNING

old_ckpt=torch.load(OLD_CLS_CKPT,map_location=DEVICE,weights_only=False)
MODEL_NAME=old_ckpt.get('model_name','tf_efficientnetv2_s.in21k_ft_in1k')
OLD_CLASS_TO_IDX=old_ckpt['class_to_idx']
KNOWN_CLASSES=[x for x,_ in sorted(OLD_CLASS_TO_IDX.items(),key=lambda kv:kv[1])]
ALL_CLASSES=KNOWN_CLASSES+['NON_CORAL']
CLASS_TO_IDX={c:i for i,c in enumerate(ALL_CLASSES)}
IDX_TO_CLASS={i:c for c,i in CLASS_TO_IDX.items()}
NCLS=len(ALL_CLASSES)
CLS_SIZE=int(old_ckpt.get('input_size',300))
MEAN=tuple(old_ckpt.get('mean',(0.5,0.5,0.5)))
STD=tuple(old_ckpt.get('std',(0.5,0.5,0.5)))

train_tf=transforms.Compose([
    transforms.Resize((CLS_SIZE,CLS_SIZE),interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(.5),transforms.RandomVerticalFlip(.08),
    transforms.RandomRotation(4,interpolation=InterpolationMode.BICUBIC),
    transforms.ColorJitter(brightness=.10,contrast=.10,saturation=.06,hue=.02),
    transforms.ToTensor(),transforms.Normalize(MEAN,STD),
])
val_tf=transforms.Compose([
    transforms.Resize((CLS_SIZE,CLS_SIZE),interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),transforms.Normalize(MEAN,STD),
])

class PairDataset(Dataset):
    def __init__(self,df,tf):
        self.df=df.reset_index(drop=True); self.tf=tf
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        c=self.tf(Image.open(r.context_path).convert('RGB'))
        m=self.tf(Image.open(r.masked_path).convert('RGB'))
        return c,m,CLASS_TO_IDX[str(r.label)]

train_pairs=pair_df[pair_df.split=='train'].copy()
val_pairs=pair_df[pair_df.split=='validation'].copy()
train_ds=PairDataset(train_pairs,train_tf)
val_ds=PairDataset(val_pairs,val_tf)

ytrain=np.array([CLASS_TO_IDX[x] for x in train_pairs.label])
counts=np.bincount(ytrain,minlength=NCLS).astype(np.float32)
class_w=np.sqrt(counts.max()/np.maximum(counts,1.)); class_w/=class_w.mean()
sampler=WeightedRandomSampler(torch.tensor(class_w[ytrain],dtype=torch.double),len(ytrain),replacement=True)

train_loader=DataLoader(train_ds,batch_size=CLS_BATCH,sampler=sampler,num_workers=2,pin_memory=True,persistent_workers=True)
val_loader=DataLoader(val_ds,batch_size=CLS_BATCH,shuffle=False,num_workers=2,pin_memory=True,persistent_workers=True)

class DualViewEfficientNet(nn.Module):
    def __init__(self,model_name,num_classes):
        super().__init__()
        self.backbone=timm.create_model(model_name,pretrained=False,num_classes=0,global_pool='avg')
        d=int(self.backbone.num_features)
        self.fusion=nn.Sequential(nn.Linear(2*d,d),nn.GELU(),nn.LayerNorm(d),nn.Dropout(.20))
        self.head=nn.Linear(d,num_classes)
        self.feat_dim=d
    def forward(self,context,masked,return_embedding=False):
        fc=self.backbone(context); fm=self.backbone(masked)
        emb=self.fusion(torch.cat([fc,fm],dim=1))
        logits=self.head(emb)
        return (logits,emb) if return_embedding else logits

dual_model=DualViewEfficientNet(MODEL_NAME,NCLS).to(DEVICE)

# Transfer every matching tensor from old single-view classifier into shared backbone.
old_state=old_ckpt['model_state_dict']
bb_state=dual_model.backbone.state_dict()
transfer={k:v for k,v in old_state.items() if k in bb_state and bb_state[k].shape==v.shape}
dual_model.backbone.load_state_dict(transfer,strict=False)
print("Transferred backbone tensors:",len(transfer),"/",len(bb_state))
print("Classes:",CLASS_TO_IDX)
print("STATUS: DUAL_MODEL_READY")


In [ ]:
# CELL 7 — TRAIN DUAL-VIEW CLASSIFIER + SAVE BEST

DUAL_EPOCHS=80
PATIENCE=18
BEST_DUAL=CLS_ROOT/'best_dualview_effnetv2s_macro_f1.pth'
LAST_DUAL=CLS_ROOT/'last_dualview_effnetv2s.pth'
HISTORY=CLS_ROOT/'training_history.csv'

criterion=nn.CrossEntropyLoss(
    weight=torch.tensor(class_w,dtype=torch.float32,device=DEVICE),
    label_smoothing=.03
)
optimizer=torch.optim.AdamW(dual_model.parameters(),lr=1e-5,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=DUAL_EPOCHS,eta_min=2e-7)
scaler=torch.amp.GradScaler('cuda')

def eval_dual(model,loader,with_emb=False):
    model.eval(); yt=[]; yp=[]; probs=[]; embs=[]
    with torch.no_grad():
        for c,m,y in loader:
            c=c.to(DEVICE,non_blocking=True); m=m.to(DEVICE,non_blocking=True)
            y=torch.as_tensor(y,device=DEVICE)
            with torch.autocast('cuda',dtype=torch.float16):
                logits,emb=model(c,m,return_embedding=True)
            p=torch.softmax(logits.float(),dim=1); pred=p.argmax(1)
            yt.extend(y.cpu().tolist()); yp.extend(pred.cpu().tolist())
            probs.append(p.cpu().numpy())
            if with_emb: embs.append(emb.float().cpu().numpy())
    yt=np.asarray(yt); yp=np.asarray(yp); probs=np.concatenate(probs)
    metrics={
        'accuracy':accuracy_score(yt,yp),
        'macro_precision':precision_score(yt,yp,average='macro',zero_division=0),
        'macro_recall':recall_score(yt,yp,average='macro',zero_division=0),
        'macro_f1':f1_score(yt,yp,average='macro',zero_division=0),
        'weighted_f1':f1_score(yt,yp,average='weighted',zero_division=0),
    }
    if with_emb: return metrics,yt,yp,probs,np.concatenate(embs)
    return metrics,yt,yp,probs

best_f1=-1.; bad=0; best_epoch=0; hist=[]
for epoch in range(1,DUAL_EPOCHS+1):
    dual_model.train(); total=0.; seen=0
    for c,m,y in train_loader:
        c=c.to(DEVICE,non_blocking=True); m=m.to(DEVICE,non_blocking=True)
        y=torch.as_tensor(y,device=DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=torch.float16):
            logits=dual_model(c,m)
            loss=criterion(logits,y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(dual_model.parameters(),5.)
        scaler.step(optimizer); scaler.update()
        total+=float(loss.detach().cpu())*c.size(0); seen+=c.size(0)
    scheduler.step()
    vm,_,_,_=eval_dual(dual_model,val_loader)
    row={'epoch':epoch,'train_loss':total/max(seen,1),'lr':optimizer.param_groups[0]['lr'],**{f'val_{k}':float(v) for k,v in vm.items()}}
    hist.append(row); pd.DataFrame(hist).to_csv(HISTORY,index=False)
    payload={
        'model_state_dict':dual_model.state_dict(),
        'model_name':MODEL_NAME,'class_to_idx':CLASS_TO_IDX,'known_classes':KNOWN_CLASSES,
        'input_size':CLS_SIZE,'context_padding':CONTEXT_PAD,'mean':MEAN,'std':STD,
        'epoch':epoch,'val_metrics':{k:float(v) for k,v in vm.items()},
        'architecture':'DualViewEfficientNetV2S',
    }
    torch.save(payload,LAST_DUAL)
    if vm['macro_f1']>best_f1:
        best_f1=float(vm['macro_f1']); best_epoch=epoch; bad=0; torch.save(payload,BEST_DUAL)
    else:
        bad+=1
    print(f"Epoch {epoch:03d} | loss={row['train_loss']:.4f} | Acc={vm['accuracy']:.4f} | MacroF1={vm['macro_f1']:.4f} | Best={best_f1:.4f}")
    if bad>=PATIENCE:
        print("Early stopping."); break

print("Best epoch:",best_epoch,"Best Macro-F1:",best_f1)
print("STATUS: DUAL_TRAINING_COMPLETE")


In [ ]:
# CELL 8 — OOD PROTOTYPES + FINAL CLASSIFIER DECISION

best_ckpt=torch.load(BEST_DUAL,map_location=DEVICE,weights_only=False)
dual_model.load_state_dict(best_ckpt['model_state_dict']).eval()
vm,vt,vp,vprob,vemb=eval_dual(dual_model,val_loader,with_emb=True)

report=pd.DataFrame(classification_report(
    vt,vp,target_names=[IDX_TO_CLASS[i] for i in range(NCLS)],
    zero_division=0,output_dict=True
)).T
report.to_csv(CLS_ROOT/'validation_report.csv')
np.savetxt(CLS_ROOT/'validation_confusion_matrix.csv',confusion_matrix(vt,vp),fmt='%d',delimiter=',')

embn=vemb/np.maximum(np.linalg.norm(vemb,axis=1,keepdims=True),1e-9)
prototypes={}; sim_thr={}
for name in KNOWN_CLASSES:
    ci=CLASS_TO_IDX[name]
    idx=np.where(vt==ci)[0]
    if not len(idx): continue
    proto=embn[idx].mean(0); proto/=max(np.linalg.norm(proto),1e-9)
    prototypes[name]=proto.astype(np.float32)
    sim_thr[name]=float(np.percentile(embn[idx]@proto,3.0))

np.savez(CAL_ROOT/'ood_prototypes.npz',**{f'proto__{k}':v for k,v in prototypes.items()})
cal={
    'known_classes':KNOWN_CLASSES,'class_to_idx':CLASS_TO_IDX,
    'similarity_thresholds':sim_thr,
    'known_conf_threshold':0.50,
    'known_margin_threshold':0.08,
    'non_coral_threshold':0.45,
    'note':'Pragmatic open-set gate; no explicit unseen-species validation set exists.'
}
(CAL_ROOT/'ood_calibration.json').write_text(json.dumps(cal,indent=2),encoding='utf-8')
print("Validation:",vm)
print("OOD thresholds:",sim_thr)
print("STATUS: OOD_READY")


In [ ]:
# CELL 9 — FINAL INFERENCE FUNCTIONS: YOLO → CORALSCOP → MASK RESOLVE → DUAL VIEW

detector=YOLO(str(DETECTOR_CKPT))
CANDIDATE_CONF=.12
DETECT_IMGSZ=960
DETECT_IOU=.70
MAX_DET=500

proto_npz=np.load(CAL_ROOT/'ood_prototypes.npz')
PROTOTYPES={k.replace('proto__',''):proto_npz[k] for k in proto_npz.files}
cal=json.loads((CAL_ROOT/'ood_calibration.json').read_text(encoding='utf-8'))

def make_views(rgb,mask,box):
    H,W=rgb.shape[:2]
    x1,y1,x2,y2=padded_box(box,W,H,CONTEXT_PAD)
    context=rgb[y1:y2,x1:x2].copy()
    lm=mask[y1:y2,x1:x2]
    masked=context.copy(); masked[~lm]=0
    return Image.fromarray(context),Image.fromarray(masked)

def classify_instances(rgb,masks,boxes):
    if not masks: return []
    cs=[]; ms=[]
    for mask,box in zip(masks,boxes):
        c,m=make_views(rgb,mask,box)
        cs.append(val_tf(c)); ms.append(val_tf(m))
    ct=torch.stack(cs).to(DEVICE); mt=torch.stack(ms).to(DEVICE)
    with torch.inference_mode(),torch.autocast('cuda',dtype=torch.float16):
        logits,emb=dual_model(ct,mt,return_embedding=True)
    probs=torch.softmax(logits.float(),1).cpu().numpy()
    emb=emb.float().cpu().numpy()
    emb/=np.maximum(np.linalg.norm(emb,axis=1,keepdims=True),1e-9)
    out=[]
    for p,e in zip(probs,emb):
        order=np.argsort(p)[::-1]; top=int(order[0]); second=int(order[1])
        raw=IDX_TO_CLASS[top]; conf=float(p[top]); margin=float(p[top]-p[second])
        non=float(p[CLASS_TO_IDX['NON_CORAL']])
        if raw=='NON_CORAL' or non>=cal['non_coral_threshold']:
            label='Non-Coral'; sim=None
        else:
            proto=PROTOTYPES.get(raw); sim=float(e@proto) if proto is not None else -1.
            if conf<cal['known_conf_threshold'] or margin<cal['known_margin_threshold'] or sim<cal['similarity_thresholds'].get(raw,-1.):
                label='Other/Unknown Coral'
            else:
                label=raw
        out.append({'label':label,'raw_top1':raw,'confidence':conf,'margin':margin,'similarity':sim,'prob_vector':p.astype(np.float32)})
    return out

def detect_refine_resolve(rgb):
    r=detector.predict(source=rgb,imgsz=DETECT_IMGSZ,conf=CANDIDATE_CONF,iou=DETECT_IOU,max_det=MAX_DET,device=0,verbose=False)[0]
    if r.boxes is None or len(r.boxes)==0: return []
    boxes=r.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
    confs=r.boxes.conf.detach().cpu().numpy().astype(np.float32)
    coralscop_predictor.set_image(rgb)
    cand=[]
    for box,dc in zip(boxes,confs):
        with torch.inference_mode():
            masks,scores,_=coralscop_predictor.predict(box=box,multimask_output=True)
        idx=int(np.argmax(scores)); mask=masks[idx].astype(bool); ms=float(scores[idx])
        if mask.sum()<MASK_MIN_PIXELS: continue
        mb=mask_bbox(mask)
        if mb is None: continue
        cand.append({'box':mb,'det_box':box,'mask':mask,'det_conf':float(dc),'mask_score':ms,'fused_score':float(math.sqrt(max(float(dc),1e-6)*max(ms,1e-6)))})
    cand.sort(key=lambda z:z['fused_score'],reverse=True)
    kept=[]
    for c in cand:
        if all(mask_iou(c['mask'],k['mask'])<MASK_DUP_IOU for k in kept):
            kept.append(c)
    return kept

print("Detector:",DETECTOR_CKPT)
print("Candidate confidence:",CANDIDATE_CONF)
print("Final classifier:",BEST_DUAL)
print("STATUS: FINAL_INFERENCE_READY")


In [ ]:
# CELL 10 — SAM 2.1 TEMPORAL MEMORY + VIDEO CONFIG

from sam2.sam2_video_predictor import SAM2VideoPredictor

SAM2_MODEL_ID='facebook/sam2-hiera-large'
KEYFRAME_STRIDE=5
MAX_SEED_GAP=15
SEED_MASK_IOU=.20
SEED_BOX_IOU=.12
MAX_SAM2_TRACKS=60
CLS_STRIDE=3
EMA_ALPHA=.25

def centroid(b):
    x1,y1,x2,y2=map(float,b)
    return np.array([(x1+x2)/2,(y1+y2)/2],dtype=np.float32)

def associate_seed_tracks(tracks,dets,fi,W,H):
    active=[tid for tid,t in tracks.items() if fi-t['last_frame']<=MAX_SEED_GAP]
    if not active or not dets: return [],list(range(len(dets)))
    cost=np.ones((len(active),len(dets)),np.float32)
    diag=max(math.hypot(W,H),1.)
    for i,tid in enumerate(active):
        for j,d in enumerate(dets):
            mi=mask_iou(tracks[tid]['last_mask'],d['mask'])
            bi=box_iou(tracks[tid]['last_box'],d['box'])
            cd=np.linalg.norm(centroid(tracks[tid]['last_box'])-centroid(d['box']))/diag
            cost[i,j]=1-max(mi,bi,max(0.,1-4*cd))
    rr,cc=linear_sum_assignment(cost)
    matches=[]; used=set()
    for r,c in zip(rr,cc):
        tid=active[r]; d=dets[c]
        if mask_iou(tracks[tid]['last_mask'],d['mask'])<SEED_MASK_IOU and box_iou(tracks[tid]['last_box'],d['box'])<SEED_BOX_IOU:
            continue
        matches.append((tid,c)); used.add(c)
    return matches,[j for j in range(len(dets)) if j not in used]

VIDEO_ZIP=MYDRIVE/'Coral_video'/'coral_video.zip'
if not VIDEO_ZIP.exists(): raise FileNotFoundError(VIDEO_ZIP)
print("SAM2:",SAM2_MODEL_ID)
print("Video ZIP:",VIDEO_ZIP)
print("STATUS: VIDEO_CONFIG_READY")


In [ ]:
# CELL 11 — FINAL VIDEO: KEYFRAME YOLO+CORALSCOP → SAM2 MEMORY → DUAL-VIEW LABELS

VIDEO_WORK=Path('/content/CoralGuardX_Video')
if VIDEO_WORK.exists(): shutil.rmtree(VIDEO_WORK)
(VIDEO_WORK/'input').mkdir(parents=True,exist_ok=True)

with zipfile.ZipFile(VIDEO_ZIP,'r') as z:
    z.extractall(VIDEO_WORK/'input')

exts={'.mp4','.mov','.avi','.mkv','.m4v','.webm','.mpg','.mpeg'}
video_paths=sorted([p for p in (VIDEO_WORK/'input').rglob('*') if p.is_file() and p.suffix.lower() in exts])
if not video_paths: raise RuntimeError("No videos found.")

def extract_frames(vp,frame_dir):
    if frame_dir.exists(): shutil.rmtree(frame_dir)
    frame_dir.mkdir(parents=True,exist_ok=True)
    cap=cv2.VideoCapture(str(vp))
    if not cap.isOpened(): raise RuntimeError(f"Could not open {vp}")
    fps=float(cap.get(cv2.CAP_PROP_FPS))
    if not np.isfinite(fps) or fps<=0: fps=25.
    W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n=0
    while True:
        ok,frame=cap.read()
        if not ok: break
        cv2.imwrite(str(frame_dir/f"{n:06d}.jpg"),frame,[int(cv2.IMWRITE_JPEG_QUALITY),95])
        n+=1
    cap.release()
    return n,fps,W,H

video_summaries=[]

for vi,vp in enumerate(video_paths,1):
    stem=vp.stem.replace(' ','_')
    out_dir=VIDEO_ROOT/stem; out_dir.mkdir(parents=True,exist_ok=True)
    final_video=out_dir/f"{stem}_CoralGuardX_FINAL.mp4"
    final_json=out_dir/f"{stem}_summary.json"

    if final_video.exists() and final_json.exists():
        print("SKIP:",vp.name)
        video_summaries.append(json.loads(final_json.read_text(encoding='utf-8')))
        continue

    print("\n"+"="*110)
    print(f"VIDEO {vi}/{len(video_paths)}: {vp.name}")

    frame_dir=VIDEO_WORK/stem/'frames'
    n_frames,fps,W,H=extract_frames(vp,frame_dir)

    # PASS A — sparse high-recall candidates + coral-specific mask refinement.
    tracks={}; next_id=1
    for fi in range(0,n_frames,KEYFRAME_STRIDE):
        bgr=cv2.imread(str(frame_dir/f"{fi:06d}.jpg"))
        rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
        dets=detect_refine_resolve(rgb)
        matches,unmatched=associate_seed_tracks(tracks,dets,fi,W,H)

        for tid,di in matches:
            d=dets[di]; t=tracks[tid]
            t['last_frame']=fi; t['last_box']=d['box']; t['last_mask']=d['mask']
            t['hits']+=1; t['score_sum']+=d['fused_score']

        for di in unmatched:
            d=dets[di]; tid=next_id; next_id+=1
            tracks[tid]={
                'id':tid,'seed_frame':fi,'seed_box':d['box'].copy(),'seed_mask':d['mask'].copy(),
                'last_frame':fi,'last_box':d['box'].copy(),'last_mask':d['mask'].copy(),
                'hits':1,'score_sum':d['fused_score'],
            }

        if fi%(KEYFRAME_STRIDE*20)==0:
            print(f"Seed pass {fi}/{n_frames} | tracks={len(tracks)}")

    track_list=list(tracks.values())
    track_list.sort(key=lambda t:(t['hits'],t['score_sum']/max(t['hits'],1)),reverse=True)
    track_list=track_list[:MAX_SAM2_TRACKS]
    if not track_list:
        raise RuntimeError(f"No coral seeds found in {vp.name}")

    print("SAM2 seed tracks:",len(track_list))

    # Free heavy detector/CoralSCOP GPU memory before SAM2 video memory.
    coralscop_predictor.reset_image()
    coralscop_model.to('cpu')
    del detector
    gc.collect(); torch.cuda.empty_cache()

    sam2_predictor=SAM2VideoPredictor.from_pretrained(SAM2_MODEL_ID)
    state=sam2_predictor.init_state(
        video_path=str(frame_dir),
        offload_video_to_cpu=True,
        offload_state_to_cpu=False,
        async_loading_frames=False
    )

    for t in track_list:
        sam2_predictor.add_new_points_or_box(
            inference_state=state,
            frame_idx=int(t['seed_frame']),
            obj_id=int(t['id']),
            box=np.asarray(t['seed_box'],dtype=np.float32),
        )

    # HQ FFmpeg writer with original audio.
    cmd=[
        'ffmpeg','-loglevel','error','-y',
        '-f','rawvideo','-vcodec','rawvideo','-pix_fmt','bgr24',
        '-s',f'{W}x{H}','-r',f'{fps:.8f}','-i','pipe:0',
        '-i',str(vp),'-map','0:v:0','-map','1:a:0?',
        '-c:v','libx264','-preset','slow','-crf','12','-pix_fmt','yuv420p',
        '-c:a','aac','-b:a','192k','-movflags','+faststart','-shortest',str(final_video)
    ]
    ff=subprocess.Popen(cmd,stdin=subprocess.PIPE,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE)

    cls_state={}
    transitions=0; switches=0; frames_done=0; propagated_masks=0

    with torch.inference_mode():
        for out_fi,out_ids,out_logits in sam2_predictor.propagate_in_video(state,start_frame_idx=0):
            fi=int(out_fi)
            bgr=cv2.imread(str(frame_dir/f"{fi:06d}.jpg"))
            rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)

            masks=[]; boxes=[]; ids=[]
            for oi,obj_id in enumerate(out_ids):
                lg=out_logits[oi]
                if lg.ndim==3: lg=lg[0]
                mask=(lg>0.).detach().cpu().numpy().astype(bool)
                if mask.sum()<MASK_MIN_PIXELS: continue
                box=mask_bbox(mask)
                if box is None: continue
                masks.append(mask); boxes.append(box); ids.append(int(obj_id))

            propagated_masks+=len(masks)

            if masks and fi%CLS_STRIDE==0:
                obs=classify_instances(rgb,masks,boxes)
                for obj_id,o in zip(ids,obs):
                    p=o['prob_vector']
                    s=cls_state.get(obj_id)
                    if s is None:
                        sm=p.copy(); s={'prob':sm,'last_label':None}
                    else:
                        sm=EMA_ALPHA*p+(1-EMA_ALPHA)*s['prob']; sm/=max(float(sm.sum()),1e-9); s['prob']=sm

                    order=np.argsort(s['prob'])[::-1]
                    top=int(order[0]); sec=int(order[1])
                    raw=IDX_TO_CLASS[top]; conf=float(s['prob'][top]); margin=float(s['prob'][top]-s['prob'][sec])
                    non=float(s['prob'][CLASS_TO_IDX['NON_CORAL']])

                    if raw=='NON_CORAL' or non>=cal['non_coral_threshold']:
                        label='Non-Coral'
                    elif conf<cal['known_conf_threshold'] or margin<cal['known_margin_threshold'] or o['label']=='Other/Unknown Coral':
                        label='Other/Unknown Coral'
                    else:
                        label=raw

                    if s['last_label'] is not None:
                        transitions+=1
                        if label!=s['last_label']: switches+=1
                    s.update({'last_label':label,'label':label,'confidence':conf})
                    cls_state[obj_id]=s

            overlay=bgr.copy()
            for obj_id,mask,box in zip(ids,masks,boxes):
                s=cls_state.get(obj_id,{'label':'Analyzing Coral','confidence':0.})
                label=s['label']; conf=float(s.get('confidence',0.))
                color=(120,120,120) if label=='Non-Coral' else (0,165,255) if label=='Other/Unknown Coral' else (0,255,0)

                overlay[mask]=(0.55*overlay[mask]+0.45*np.asarray(color,dtype=np.float32)).astype(np.uint8)
                contours,_=cv2.findContours(mask.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(bgr,contours,-1,color,3)

                x1,y1,x2,y2=[int(round(z)) for z in box]
                cv2.putText(
                    bgr,f"ID {obj_id} | {label} | {conf:.2f}",
                    (x1,max(25,y1-8)),cv2.FONT_HERSHEY_SIMPLEX,.72,color,2,cv2.LINE_AA
                )

            frame_out=cv2.addWeighted(overlay,.45,bgr,.55,0)
            try:
                ff.stdin.write(frame_out.tobytes())
            except BrokenPipeError:
                ff.stdin.close(); ff.wait()
                err=ff.stderr.read().decode('utf-8',errors='ignore')
                raise RuntimeError(err[-5000:])

            frames_done+=1
            if frames_done%100==0:
                print(f"Render {frames_done}/{n_frames} | masks={propagated_masks}")

    ff.stdin.close(); ff.wait()
    err=ff.stderr.read().decode('utf-8',errors='ignore')
    if ff.returncode!=0: raise RuntimeError(err[-5000:])

    summary={
        'video':vp.name,'frames':n_frames,'seed_tracks_found':len(tracks),
        'sam2_tracks_used':len(track_list),'propagated_masks':propagated_masks,
        'label_transitions':transitions,'label_switches':switches,
        'species_flicker_rate':switches/transitions if transitions else 0.,
        'detector_checkpoint':str(DETECTOR_CKPT),
        'classifier_checkpoint':str(BEST_DUAL),
        'coralscop_checkpoint':str(CORALSCOP_CKPT),
        'sam2_model':SAM2_MODEL_ID,
    }
    final_json.write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
    video_summaries.append(summary)
    print("Saved:",final_video)

    del sam2_predictor,state
    gc.collect(); torch.cuda.empty_cache()

    # Reload detector and move CoralSCOP back for the next video's seed pass.
    coralscop_model.to(DEVICE)
    detector=YOLO(str(DETECTOR_CKPT))

print("STATUS: FINAL_VIDEO_PIPELINE_COMPLETE")


In [ ]:
# CELL 12 — FINAL PROJECT SUMMARY

summary_df=pd.DataFrame(video_summaries)
summary_csv=VIDEO_ROOT/'CoralGuardX_all_video_summary.csv'
summary_df.to_csv(summary_csv,index=False)

final_summary={
    'architecture':(
        'YOLO26m High-Recall Detection -> CoralSCOP Box-Prompted Mask Refinement -> '
        'Mask-IoU Overlap Resolver -> Dual-View EfficientNetV2-S -> '
        'Embedding OOD Gate -> SAM2.1 Temporal Mask Memory'
    ),
    'detector_checkpoint':str(DETECTOR_CKPT),
    'coralscop_checkpoint':str(CORALSCOP_CKPT),
    'dual_view_classifier':str(BEST_DUAL),
    'sam2_model':SAM2_MODEL_ID,
    'known_species':KNOWN_CLASSES,
    'extra_outputs':['NON_CORAL','Other/Unknown Coral'],
    'best_classifier_macro_f1':float(best_ckpt['val_metrics']['macro_f1']),
    'videos_processed':int(len(summary_df)),
    'held_out_detector_test_used':False,
    'limitation':'Dead-coral and algae semantic labels are not explicitly trained because the current dataset does not provide those labels.'
}
final_json=FINAL_ROOT/'CoralGuardX_FINAL_PROJECT_SUMMARY.json'
final_json.write_text(json.dumps(final_summary,indent=2,ensure_ascii=False),encoding='utf-8')

print("="*115)
print("CORALGUARD-X COMPLETE")
print("Root:",FINAL_ROOT)
print("Classifier:",BEST_DUAL)
print("Videos:",VIDEO_ROOT)
print("Summary:",final_json)
print("Held-out detector TEST used: False")
print("STATUS: COMPLETE")


## Google Drive outputs

```text
MyDrive/inferance/CoralGuard_X_FINAL/
├── 01_CoralSCOP_Masks/
├── 02_DualView_Data/
├── 03_DualView_Classifier/
│   └── best_dualview_effnetv2s_macro_f1.pth
├── 04_OOD_Calibration/
│   ├── ood_prototypes.npz
│   └── ood_calibration.json
├── 05_Final_Videos/
└── CoralGuardX_FINAL_PROJECT_SUMMARY.json
```

The core final logic is:

```text
YOLO: possible coral here
  ↓
CoralSCOP: exact coral pixels
  ↓
Mask resolver: keep overlapping instances separate when their masks differ
  ↓
Dual-view classifier: morphology + local context
  ↓
OOD gate: refuse weak/unfamiliar forced labels
  ↓
SAM2.1: carry the mask/ID through time
```
